In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [2]:
# 7 habits of highly effective people
with open('html_0702.md', 'r', encoding='utf-8') as f:
    text = f.read()
    #print(text)
print(len(text))


64487


In [3]:
# here are all the unique characters that occur in this text
print(set(text))
print(list(set(text)))
print(sorted(list(set(text))))
print(len(sorted(list(set(text)))))
chars = sorted(list(set(text)))
vocab_size = len(chars)

# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(stoi['\n'])
print(encode(['A','B','C']))
print(decode(encode(['A','B','C'])))

{'F', 'y', '>', '📋', ' ', '(', '9', 'd', 'A', '_', 'T', 'M', '.', 'z', '📦', '|', 'D', 'h', '/', 'B', '0', 'v', '4', 'W', ']', '\n', 'Y', 'V', 'Q', '📝', '⚪', '*', '?', '$', 'J', '6', '&', 'U', 'q', 'C', '🔍', 'O', 'S', '#', '\\', '=', 'g', '{', '—', '1', 'R', 'H', '`', 's', 'G', '2', 'a', 'r', 'L', 'X', '🔷', 'b', 'j', 'x', 'c', '7', 'E', ',', '🔄', '-', 'P', 'N', 'e', '@', '💬', 't', 'l', 'p', '▶', 'w', '[', '📥', ':', '!', 'i', '"', '5', 'I', 'u', '📄', 'n', '}', 'm', 'K', 'f', '8', ';', 'o', '%', 'k', '+', "'", ')', '3', '<'}
['F', 'y', '>', '📋', ' ', '(', '9', 'd', 'A', '_', 'T', 'M', '.', 'z', '📦', '|', 'D', 'h', '/', 'B', '0', 'v', '4', 'W', ']', '\n', 'Y', 'V', 'Q', '📝', '⚪', '*', '?', '$', 'J', '6', '&', 'U', 'q', 'C', '🔍', 'O', 'S', '#', '\\', '=', 'g', '{', '—', '1', 'R', 'H', '`', 's', 'G', '2', 'a', 'r', 'L', 'X', '🔷', 'b', 'j', 'x', 'c', '7', 'E', ',', '🔄', '-', 'P', 'N', 'e', '@', '💬', 't', 'l', 'p', '▶', 'w', '[', '📥', ':', '!', 'i', '"', '5', 'I', 'u', '📄', 'n', '}', 'm', 'K',

In [10]:
import re
from html.parser import HTMLParser

# Step 1: Read HTML File
with open('html_0702.md', 'r', encoding='utf-8') as f:
    text = f.read()

# Step 2: Use lexer to tokenize HTML content into atomic tokens
class HTMLLexer(HTMLParser):
    def __init__(self):
        super().__init__()
        self.tokens = []

    def handle_starttag(self, tag, attrs):
        self.tokens.append(f"START_TAG: <{tag}>")
        for attr in attrs:
            self.tokens.append(f"ATTRIBUTE: {attr[0]} = {attr[1]}")
    
    def handle_endtag(self, tag):
        self.tokens.append(f"END_TAG: </{tag}>")
    
    def handle_startendtag(self, tag, attrs):
        self.tokens.append(f"SELF_CLOSING_TAG: <{tag} />")
        for attr in attrs:
            self.tokens.append(f"ATTRIBUTE: {attr[0]} = {attr[1]}")
    
    def handle_data(self, data):
        data = data.strip()
        if data:
            self.tokens.append(f"TEXT: {data}")
    
    def tokenize(self, html):
        self.feed(html)
        return self.tokens

# Step 3: Tokenize the HTML text
lexer = HTMLLexer()
tokens = lexer.tokenize(text)

print("=============================")
# Step 4: Print the atomic tokens
print(len(tokens))
print(tokens)
#for token in tokens:
#    print(token)
    

285
['TEXT: # LLM Training Data: DOM Node Properties - nodeType, nodeName, nodeValue\n\n**Generated:** Thursday, March 5th, 2026 — 7:02 AM  \n**Topics:** dom_nodetype, dom_nodename, dom_nodevalue  \n**Lines:** 520+ lines with comprehensive Chain-of-Thought\n\n---\n\n## Instruction\n\nBuild a production-ready DOM inspection and visualization tool that demonstrates the core DOM node properties: `nodeType`, `nodeName`, and `nodeValue`. This tool should help developers understand the DOM tree structure by providing:\n\n1. **Interactive DOM Tree Visualization** - Display any webpage\'s DOM structure with expandable nodes\n2. **Property Inspector** - Show real-time nodeType, nodeName, and nodeValue for selected elements\n3. **Educational Examples** - Include pre-built examples showing different node types\n4. **Search & Filter** - Allow filtering nodes by type, name, or value\n5. **Export Capability** - Export DOM structure as JSON or formatted text\n\nThe implementation must demonstrate bes

In [14]:
import re
from html.parser import HTMLParser
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace

# Step 1: Load BPE Tokenizer (You can train your own tokenizer or use a pretrained one)
# Here, we will use a basic tokenizer for demonstration purposes.
# You can train a tokenizer on a corpus or use an existing one if you have one.

tokenizer = Tokenizer(BPE())
tokenizer.pre_tokenizer = Whitespace()

# Alternatively, load a pretrained tokenizer (e.g., GPT-2 tokenizer)
# tokenizer = Tokenizer.from_pretrained('gpt2')

# Step 2: Read HTML File
with open('html_0702.md', 'r', encoding='utf-8') as f:
    text = f.read()

# Step 3: Define HTMLLexer that keeps tags intact and applies BPE to text
class HTMLLexer(HTMLParser):
    def __init__(self):
        super().__init__()
        self.tokens = []

    def handle_starttag(self, tag, attrs):
        self.tokens.append(f"START_TAG: <{tag}>")
        for attr in attrs:
            self.tokens.append(f"ATTRIBUTE: {attr[0]} = {attr[1]}")
    
    def handle_endtag(self, tag):
        self.tokens.append(f"END_TAG: </{tag}>")
    
    def handle_startendtag(self, tag, attrs):
        self.tokens.append(f"SELF_CLOSING_TAG: <{tag} />")
        for attr in attrs:
            self.tokens.append(f"ATTRIBUTE: {attr[0]} = {attr[1]}")
    
    def handle_data(self, data):
        data = data.strip()
        if data:
            # Apply BPE tokenization to the text
            bpe_tokens = tokenizer.encode(data).tokens
            self.tokens.append(f"TEXT: {' '.join(bpe_tokens)}")
    
    def tokenize(self, html):
        self.feed(html)
        return self.tokens

# Step 4: Tokenize the HTML text
lexer = HTMLLexer()
tokens = lexer.tokenize(text)

print("=============================")
print(len(tokens))
print(tokens)

# Step 5: Print the atomic tokens
for token in tokens:
    print(token)



285
['TEXT: ', 'START_TAG: <html>', 'ATTRIBUTE: lang = en', 'START_TAG: <head>', 'START_TAG: <meta>', 'ATTRIBUTE: charset = UTF-8', 'START_TAG: <meta>', 'ATTRIBUTE: name = viewport', 'ATTRIBUTE: content = width=device-width, initial-scale=1.0', 'START_TAG: <title>', 'TEXT: ', 'END_TAG: </title>', 'START_TAG: <style>', 'TEXT: ', 'END_TAG: </style>', 'END_TAG: </head>', 'START_TAG: <body>', 'START_TAG: <div>', 'ATTRIBUTE: class = container', 'START_TAG: <header>', 'ATTRIBUTE: class = header', 'ATTRIBUTE: role = banner', 'START_TAG: <h1>', 'TEXT: ', 'END_TAG: </h1>', 'START_TAG: <p>', 'TEXT: ', 'END_TAG: </p>', 'END_TAG: </header>', 'START_TAG: <div>', 'ATTRIBUTE: class = stats-bar', 'ATTRIBUTE: role = region', 'ATTRIBUTE: aria-label = DOM Statistics', 'START_TAG: <div>', 'ATTRIBUTE: class = stat-item', 'START_TAG: <div>', 'ATTRIBUTE: class = stat-value', 'ATTRIBUTE: id = totalNodes', 'TEXT: ', 'END_TAG: </div>', 'START_TAG: <div>', 'ATTRIBUTE: class = stat-label', 'TEXT: ', 'END_TAG: </d

In [12]:
!pip install tokenizers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 2.6 MB/s  0:00:01m 2.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 20.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 8.2 MB/s  0:00:00m 20.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [tokenizers]0m 10/11 [tokenizers]


In [17]:
import re
from html.parser import HTMLParser

# Step 1: Read HTML File
with open('html_0702.md', 'r', encoding='utf-8') as f:
    text = f.read()

# Step 2: Define HTMLLexer that keeps tags intact and tokenizes each word in the text
class HTMLLexer(HTMLParser):
    def __init__(self):
        super().__init__()
        self.tokens = []

    def handle_starttag(self, tag, attrs):
        self.tokens.append(f"START_TAG: <{tag}>")
        for attr in attrs:
            self.tokens.append(f"ATTRIBUTE: {attr[0]} = {attr[1]}")
    
    def handle_endtag(self, tag):
        self.tokens.append(f"END_TAG: </{tag}>")
    
    def handle_startendtag(self, tag, attrs):
        self.tokens.append(f"SELF_CLOSING_TAG: <{tag} />")
        for attr in attrs:
            self.tokens.append(f"ATTRIBUTE: {attr[0]} = {attr[1]}")
    
    def handle_data(self, data):
        data = data.strip()
        if data:
            # Tokenize the text content into words (tokens)
            words = data.split()
            for word in words:
                self.tokens.append(f"TEXT: {word}")
    
    def tokenize(self, html):
        self.feed(html)
        return self.tokens

# Step 3: Tokenize the HTML text
lexer = HTMLLexer()
tokens = lexer.tokenize(text)

print("=============================")
print(len(tokens))
print(tokens)

# Step 4: Print the atomic tokens
for token in tokens:
    print(token)


4889
['TEXT: #', 'TEXT: LLM', 'TEXT: Training', 'TEXT: Data:', 'TEXT: DOM', 'TEXT: Node', 'TEXT: Properties', 'TEXT: -', 'TEXT: nodeType,', 'TEXT: nodeName,', 'TEXT: nodeValue', 'TEXT: **Generated:**', 'TEXT: Thursday,', 'TEXT: March', 'TEXT: 5th,', 'TEXT: 2026', 'TEXT: —', 'TEXT: 7:02', 'TEXT: AM', 'TEXT: **Topics:**', 'TEXT: dom_nodetype,', 'TEXT: dom_nodename,', 'TEXT: dom_nodevalue', 'TEXT: **Lines:**', 'TEXT: 520+', 'TEXT: lines', 'TEXT: with', 'TEXT: comprehensive', 'TEXT: Chain-of-Thought', 'TEXT: ---', 'TEXT: ##', 'TEXT: Instruction', 'TEXT: Build', 'TEXT: a', 'TEXT: production-ready', 'TEXT: DOM', 'TEXT: inspection', 'TEXT: and', 'TEXT: visualization', 'TEXT: tool', 'TEXT: that', 'TEXT: demonstrates', 'TEXT: the', 'TEXT: core', 'TEXT: DOM', 'TEXT: node', 'TEXT: properties:', 'TEXT: `nodeType`,', 'TEXT: `nodeName`,', 'TEXT: and', 'TEXT: `nodeValue`.', 'TEXT: This', 'TEXT: tool', 'TEXT: should', 'TEXT: help', 'TEXT: developers', 'TEXT: understand', 'TEXT: the', 'TEXT: DOM', 'TEX

In [18]:
import re
from html.parser import HTMLParser

# Step 1: Read HTML File
with open('html_0702.md', 'r', encoding='utf-8') as f:
    text = f.read()

# Step 2: Define HTMLLexer that keeps tags intact and avoids duplicate word tokens
class HTMLLexer(HTMLParser):
    def __init__(self):
        super().__init__()
        self.tokens = []
        self.seen_words = set()  # Track seen words

    def handle_starttag(self, tag, attrs):
        self.tokens.append(f"START_TAG: <{tag}>")
        for attr in attrs:
            self.tokens.append(f"ATTRIBUTE: {attr[0]} = {attr[1]}")
    
    def handle_endtag(self, tag):
        self.tokens.append(f"END_TAG: </{tag}>")
    
    def handle_startendtag(self, tag, attrs):
        self.tokens.append(f"SELF_CLOSING_TAG: <{tag} />")
        for attr in attrs:
            self.tokens.append(f"ATTRIBUTE: {attr[0]} = {attr[1]}")
    
    def handle_data(self, data):
        data = data.strip()
        if data:
            # Tokenize the text content into words (tokens)
            words = data.split()
            for word in words:
                if word.lower() not in self.seen_words:  # Ignore repeated words
                    self.tokens.append(f"TEXT: {word}")
                    self.seen_words.add(word.lower())  # Mark word as seen
    
    def tokenize(self, html):
        self.feed(html)
        return self.tokens

# Step 3: Tokenize the HTML text
lexer = HTMLLexer()
tokens = lexer.tokenize(text)

print("=============================")
print(len(tokens))
print(tokens)

# Step 4: Print the atomic tokens
for token in tokens:
    print(token)


1902
['TEXT: #', 'TEXT: LLM', 'TEXT: Training', 'TEXT: Data:', 'TEXT: DOM', 'TEXT: Node', 'TEXT: Properties', 'TEXT: -', 'TEXT: nodeType,', 'TEXT: nodeName,', 'TEXT: nodeValue', 'TEXT: **Generated:**', 'TEXT: Thursday,', 'TEXT: March', 'TEXT: 5th,', 'TEXT: 2026', 'TEXT: —', 'TEXT: 7:02', 'TEXT: AM', 'TEXT: **Topics:**', 'TEXT: dom_nodetype,', 'TEXT: dom_nodename,', 'TEXT: dom_nodevalue', 'TEXT: **Lines:**', 'TEXT: 520+', 'TEXT: lines', 'TEXT: with', 'TEXT: comprehensive', 'TEXT: Chain-of-Thought', 'TEXT: ---', 'TEXT: ##', 'TEXT: Instruction', 'TEXT: Build', 'TEXT: a', 'TEXT: production-ready', 'TEXT: inspection', 'TEXT: and', 'TEXT: visualization', 'TEXT: tool', 'TEXT: that', 'TEXT: demonstrates', 'TEXT: the', 'TEXT: core', 'TEXT: properties:', 'TEXT: `nodeType`,', 'TEXT: `nodeName`,', 'TEXT: `nodeValue`.', 'TEXT: This', 'TEXT: should', 'TEXT: help', 'TEXT: developers', 'TEXT: understand', 'TEXT: tree', 'TEXT: structure', 'TEXT: by', 'TEXT: providing:', 'TEXT: 1.', 'TEXT: **Interactive